# الفصل 17: المكوّنات الحديثة: RMSNorm وSwiGLU

> بعد كل الابتكارات الكبرى التي استعرضناها (RoPE، GQA، MoE، Mamba)، هذا الفصل يُركّز على تفاصيل أصغر حجمًا لكن شائعة بشكل شبه كوني في كل نموذج حديث تقريبًا (LLaMA، Mistral، Qwen، Gemma): استبدال LayerNorm...

- **الجزء:** 4 — المعماريات الحديثة
- **الفصل:** 17
- **المستوى:** متقدم
- **زمن القراءة:** 20 دقيقة
- **المتطلبات:** الفصل 16: نماذج الفضاء الحالة: SSM وMamba
- **الوسوم:** RMSNorm، SwiGLU، GLU، LLaMA

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch17-modern-components.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-04-modern-architectures%2Fch17-modern-components.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-04-modern-architectures/ch17-modern-components.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch17-modern-components.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch17-modern-components.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch17-modern-components.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch17-modern-components.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-04-modern-architectures%2Fch17-modern-components.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-04-modern-architectures/ch17-modern-components.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch17-modern-components.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch17-modern-components.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch17-modern-components.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> بعد كل الابتكارات الكبرى التي استعرضناها (RoPE، GQA، MoE، Mamba)، هذا الفصل يُركّز على تفاصيل أصغر حجمًا لكن شائعة بشكل شبه كوني في كل نموذج حديث تقريبًا (LLaMA، Mistral، Qwen، Gemma): استبدال LayerNorm...

> **بيانات الفصل**
>
> - **الجزء:** 4 — المعماريات الحديثة
> - **المستوى:** متقدم
> - **زمن القراءة:** 20 دقيقة
> - **المتطلبات:** الفصل 16: نماذج الفضاء الحالة: SSM وMamba


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم SwiGLU بالتفصيل
- فهم RMSNorm
- فهم SwiGLU


## 1. لماذا يهمّك هذا الموضوع؟

بعد كل الابتكارات الكبرى التي استعرضناها (RoPE، GQA، MoE، Mamba)، هذا الفصل يُركّز على تفاصيل أصغر حجمًا لكن **شائعة بشكل شبه كوني** في كل نموذج حديث تقريبًا (LLaMA، Mistral، Qwen، Gemma): استبدال **LayerNorm** بـ**RMSNorm** الأبسط والأسرع، واستبدال **ReLU/GELU** التقليدية في طبقات FeedForward بـ**SwiGLU**. هذه التحسينات قد تبدو صغيرة على الورق، لكنها مُجتمعة توفّر تحسينات ملموسة في سرعة التدريب والاستدلال وجودة النموذج النهائية — ولهذا تبنّتها كل الأوراق المرجعية الحديثة تقريبًا دون استثناء تقريبًا.

---


## 2. الخريطة الذهنية


```text
RMSNorm: تبسيط LayerNorm
│
├── LayerNorm يُطبّع بطرح المتوسط والقسمة على الانحراف المعياري
├── RMSNorm يتجاهل طرح المتوسط، ويُطبّع فقط بجذر متوسط مربعات القيم
│   RMS(x) = sqrt(mean(x²))
│   y = (x / RMS(x)) * γ
│
└── الفائدة: حساب أبسط وأسرع (عملية حسابية أقل)،
    مع أداء مماثل تجريبيًا تقريبًا للـ LayerNorm الكامل

SwiGLU: تطوير على طبقة FeedForward التقليدية
│
├── FFN التقليدية: Linear → Activation (ReLU/GELU) → Linear
│
├── GLU (Gated Linear Unit): فكرة "البوابة" الضربية
│   GLU(x) = (xW_1) ⊗ σ(xW_2)      (⊗ = ضرب عنصر بعنصر)
│
├── SwiGLU: استبدال σ (sigmoid) بـ Swish/SiLU
│   SwiGLU(x) = (xW_1) ⊗ SiLU(xW_2)
│   ثم: FFN_SwiGLU(x) = (SwiGLU(x)) W_3
│
└── يتطلب 3 مصفوفات أوزان بدلًا من 2 (W_1, W_2, W_3)
    لذا تُقلَّص أبعاد d_ff عادة بعامل 2/3 للحفاظ على نفس عدد المعاملات الكلي

المعمارية المرجعية الحديثة (LLaMA-style):
Input → RMSNorm → Attention (RoPE + GQA) → +residual
      → RMSNorm → SwiGLU FFN → +residual → Output
```


---


## 3. الشرح المتدرج

### 🟡 طبقة الآلية: RMSNorm بالتفصيل

**LayerNorm (تذكير من الفصل 11)**:

$$\text{LN}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

**RMSNorm (Zhang & Sennrich, 2019)**:

$$\text{RMS}(x) = \sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}$$

$$\text{RMSNorm}(x) = \gamma \cdot \frac{x}{\text{RMS}(x)}$$

**الفروقات الجوهرية**:
1. **لا طرح للمتوسط ($\mu$)**: RMSNorm يفترض أن إعادة التمركز (re-centering) حول الصفر ليست ضرورية بنفس أهمية إعادة القياس (re-scaling)؛ الأبحاث التجريبية أظهرت أن معظم فائدة التطبيع تأتي من التحكم في المقياس (scale) وليس من طرح المتوسط تحديدًا.
2. **لا معامل $\beta$ (الإزاحة)**: فقط $\gamma$ (التحجيم) يبقى كمعامل قابل للتعلم.
3. **حساب أبسط وأسرع**: تجنب حساب المتوسط يُقلِّل عدد العمليات الحسابية (خصوصًا مهم عند تكرار هذه العملية مليارات المرات عبر طبقات نموذج ضخم ومليارات الرموز أثناء التدريب).

**لماذا ينجح رغم تبسيطه؟** التفسير الأكثر قبولًا هو أن الفائدة الأساسية لأي تقنية تطبيع هي **استقرار مقياس (scale) التنشيطات** عبر الطبقات العميقة (منع الانفجار أو التلاشي)، وهذا تحديدًا ما تضمنه القسمة على RMS — أما إعادة التمركز حول الصفر (طرح المتوسط) فتبيّن تجريبيًا أنها أقل أهمية بكثير لتحقيق هذا الاستقرار.

### 🟡 SwiGLU بالتفصيل

**الخلفية — عائلة Gated Linear Units (GLU)**:

فكرة GLU الأساسية (Dauphin et al., 2017) هي استخدام "بوابة" ضربية تتحكم في تدفق المعلومات:

$$\text{GLU}(x, W, V) = \sigma(xW) \otimes (xV)$$

حيث $\sigma$ دالة سيجمويد، و$\otimes$ ضرب عنصر بعنصر (element-wise). الفكرة: جزء من الشبكة ($xV$) يحمل "المحتوى"، بينما جزء آخر ($\sigma(xW)$) يحمل "قرار البوابة" الذي يُحدِّد كم من هذا المحتوى يمر فعليًا.

**SwiGLU (Shazeer, 2020)**: استبدال دالة السيجمويد ببوابة Swish/SiLU:

$$\text{SiLU}(x) = x \cdot \sigma(x)$$

$$\text{SwiGLU}(x, W, V) = \text{SiLU}(xW) \otimes (xV)$$

ثم يُطبَّق إسقاط خطي أخير $W_2$ على ناتج SwiGLU ليعود للبُعد الأصلي:

$$\text{FFN}_{\text{SwiGLU}}(x) = (\text{SiLU}(xW_1) \otimes (xW_3)) W_2$$

**ملاحظة حول عدد المعاملات**: بما أن SwiGLU يحتاج ثلاث مصفوفات ($W_1, W_2, W_3$) بدلًا من اثنتين فقط في FFN التقليدية، فإن الممارسة الشائعة (كما في LLaMA) هي تقليص البُعد الداخلي $d_{ff}$ من القيمة التقليدية (4 أضعاف $d_{model}$) إلى تقريبًا $\frac{2}{3} \times 4 \times d_{model}$، للحفاظ على نفس إجمالي عدد المعاملات تقريبًا مقارنة بـ FFN التقليدية ذات البعد الكامل.

**لماذا SwiGLU يتفوق تجريبيًا؟** ورقة Shazeer (2020) لم تُقدِّم تفسيرًا نظريًا قاطعًا (اعترف المؤلف صراحة أن "نُرجِّح نجاحها إلى الرحمة الإلهية" بطريقة فكاهية مشهورة في الورقة!)، لكن التجارب الواسعة عبر عشرات الأوراق اللاحقة أكّدت تحسينات ثابتة ومتسقة في جودة النماذج عند استخدام SwiGLU مقارنة بـ ReLU أو GELU البسيطة، في كل مقاييس الحجم المُختبَرة تقريبًا.

### 🔴 طبقة التعمّق: الكود الكامل


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# 1. RMSNorm من الصفر
# ============================================================
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # RMS(x) = sqrt(mean(x²) + eps)
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.gamma

# ============================================================
# 2. SwiGLU FeedForward Network من الصفر
# ============================================================
class SwiGLUFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int = None, dropout: float = 0.1):
        super().__init__()
        if d_ff is None:
            # الصيغة القياسية في LLaMA: 2/3 من 4×d_model، مقرَّبة لأقرب مضاعف لـ 256
            d_ff = int(2 * (4 * d_model) / 3)
            d_ff = 256 * ((d_ff + 255) // 256)

        self.W1 = nn.Linear(d_model, d_ff, bias=False)   # بوابة SiLU
        self.W3 = nn.Linear(d_model, d_ff, bias=False)   # المحتوى
        self.W2 = nn.Linear(d_ff, d_model, bias=False)   # الإسقاط النهائي
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        gate = F.silu(self.W1(x))       # SiLU(xW1)
        content = self.W3(x)            # xW3
        hidden = gate * content         # الضرب العنصري (البوابة)
        return self.dropout(self.W2(hidden))

# ============================================================
# 3. ModernTransformerBlock — المعمارية المرجعية لنماذج 2023+
#    (يدمج كل مكونات الجزء الرابع: RoPE + GQA + RMSNorm + SwiGLU)
# ============================================================
class ModernTransformerBlock(nn.Module):
    """
    كتلة Transformer حديثة بأسلوب LLaMA/Mistral/Qwen:
    - RMSNorm بدلاً من LayerNorm
    - GQA مع RoPE بدلاً من MHA التقليدي
    - SwiGLU بدلاً من ReLU/GELU FFN
    - Pre-Norm (كما في الفصل 11)
    """
    def __init__(self, d_model, num_heads, num_kv_groups, d_ff=None,
                 max_seq_len=8192, dropout=0.0):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GroupedQueryAttention(   # من الفصل 14
            d_model, num_heads, num_kv_groups, max_seq_len, dropout
        )
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLUFeedForward(d_model, d_ff, dropout)

    def forward(self, x, mask=None):
        x = x + self.attn(self.attn_norm(x), mask)
        x = x + self.ffn(self.ffn_norm(x))
        return x

class ModernLanguageModel(nn.Module):
    """نموذج لغوي كامل بالمعمارية الحديثة المرجعية (أسلوب LLaMA)."""
    def __init__(self, vocab_size, d_model=4096, num_layers=32,
                 num_heads=32, num_kv_groups=8, d_ff=None, max_seq_len=8192):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            ModernTransformerBlock(d_model, num_heads, num_kv_groups, d_ff, max_seq_len)
            for _ in range(num_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids, targets=None):
        batch_size, seq_len = input_ids.shape
        x = self.embedding(input_ids)

        causal_mask = create_causal_mask(seq_len).to(input_ids.device)  # من الفصل 10

        for layer in self.layers:
            x = layer(x, causal_mask)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                     targets.view(-1), ignore_index=-100)
        return logits, loss

# ============================================================
# 4. مقارنة عملية لسرعة RMSNorm مقابل LayerNorm
# ============================================================
def compare_norm_speed(d_model=4096, batch_size=32, seq_len=2048,
                         num_iterations=1000, device='cuda'):
    import time

    x = torch.randn(batch_size, seq_len, d_model, device=device)
    layer_norm = nn.LayerNorm(d_model).to(device)
    rms_norm = RMSNorm(d_model).to(device)

    # تسخين (warmup) لضمان قياس دقيق
    for _ in range(10):
        _ = layer_norm(x)
        _ = rms_norm(x)
    torch.cuda.synchronize()

    start = time.time()
    for _ in range(num_iterations):
        _ = layer_norm(x)
    torch.cuda.synchronize()
    layer_norm_time = time.time() - start

    start = time.time()
    for _ in range(num_iterations):
        _ = rms_norm(x)
    torch.cuda.synchronize()
    rms_norm_time = time.time() - start

    print(f"LayerNorm: {layer_norm_time*1000:.2f}ms لـ {num_iterations} تكرار")
    print(f"RMSNorm:   {rms_norm_time*1000:.2f}ms لـ {num_iterations} تكرار")
    print(f"نسبة التسريع: {layer_norm_time/rms_norm_time:.2f}x")


---


## 4. مثال محلول خطوة بخطوة

لنحسب يدويًا RMSNorm لمتجه بسيط $x = [3, -4, 0, 1]$ ($d=4$)، بافتراض $\gamma = [1,1,1,1]$ و$\epsilon$ مُهمَلة لتبسيط الحساب.

**الخطوة 1 — حساب متوسط المربعات**:
$$\text{mean}(x^2) = \frac{3^2 + (-4)^2 + 0^2 + 1^2}{4} = \frac{9+16+0+1}{4} = \frac{26}{4} = 6.5$$

**الخطوة 2 — حساب RMS**:
$$\text{RMS}(x) = \sqrt{6.5} \approx 2.5495$$

**الخطوة 3 — التطبيع**:
$$\text{RMSNorm}(x) = \frac{[3, -4, 0, 1]}{2.5495} \approx [1.1768, -1.5690, 0, 0.3923]$$

**مقارنة مع LayerNorm لنفس المتجه** (للتوضيح):

المتوسط: $\mu = \frac{3-4+0+1}{4} = 0$ (بالصدفة يساوي صفرًا هنا)

التباين: $\sigma^2 = \frac{(3-0)^2+(-4-0)^2+(0-0)^2+(1-0)^2}{4} = \frac{9+16+0+1}{4} = 6.5$

$$\text{LN}(x) = \frac{[3,-4,0,1] - 0}{\sqrt{6.5}} \approx [1.1768, -1.5690, 0, 0.3923]$$

**ملاحظة مثيرة للاهتمام**: في هذا المثال بالذات، النتيجتان متطابقتان تمامًا! هذا لأن متوسط $x$ كان صفرًا بالصدفة (3-4+0+1=0)، مما يجعل RMSNorm وLayerNorm متطابقين رياضيًا في هذه الحالة الخاصة (لأن $\text{RMS}(x) = \sigma$ فقط عندما $\mu=0$). في الحالة العامة حيث $\mu \neq 0$، ستختلف النتيجتان، لكن الفرق عمليًا يكون صغيرًا نسبيًا في معظم توزيعات التنشيطات الفعلية في الشبكات العميقة، مما يُفسِّر لماذا لا يُضحّي RMSNorm بالأداء رغم تبسيطه.

---


## 5. نصائح وحيل

> 💡 **RMSNorm وSwiGLU الخيار الافتراضي لأي نموذج جديد اليوم**: إذا كنت تبني نموذجًا من الصفر في 2024 وما بعده، ابدأ بهذا المزيج مباشرة (كما في كل نماذج LLaMA/Mistral/Qwen/Gemma) بدلًا من LayerNorm وGELU التقليديين، إلا لسبب بحثي محدد يستدعي خلاف ذلك.

> 💡 **اضبط $d_{ff}$ بعناية عند استخدام SwiGLU**: تذكّر أن SwiGLU يحتاج مصفوفة وزن إضافية مقارنة بـ FFN التقليدية — طبّق تصحيح 2/3 (أو نسبة مشابهة) للحفاظ على ميزانية المعاملات الكلية مقارنة بخط الأساس.

> 💡 **قِس التسريع الفعلي على عتادك**: الفرق بين RMSNorm وLayerNorm قد يكون صغيرًا نسبيًا على معالج واحد، لكنه يتراكم بشكل كبير جدًا عبر آلاف الاستدعاءات في نموذج بعشرات الطبقات مُدرَّب على تريليونات الرموز — استخدم `compare_norm_speed` لقياس الفرق الفعلي في بيئتك.

---


## 6. أخطاء شائعة ومصائد

> ⚠️ **نسيان ضبط $d_{ff}$ عند التبديل من FFN التقليدية إلى SwiGLU**: استخدام نفس $d_{ff}$ (4×d_model) دون تصحيح عند الانتقال لـ SwiGLU يُنتج نموذجًا بعدد معاملات أكبر بكثير من المخطط له، مما قد يُشوّه مقارنات عادلة بين معماريات مختلفة.

> ⚠️ **الاعتقاد بأن RMSNorm "أفضل دائمًا" من LayerNorm في كل سياق**: التفوق التجريبي موثّق جيدًا لنماذج اللغة الكبيرة تحديدًا، لكن هذا لا يعني تلقائيًا أنه الخيار الأمثل في كل تطبيق للتعلم العميق (مثل بعض معماريات الرؤية الحاسوبية حيث لا يزال BatchNorm أو LayerNorm التقليدي شائعًا ومبررًا).

> ⚠️ **تجاهل $\epsilon$ في RMSNorm**: رغم بساطة الصيغة، لا يزال $\epsilon$ ضروريًا لتجنب القسمة على صفر في الحالة النادرة لمتجه بكل عناصره أصفارًا تمامًا (قد يحدث نظريًا في بدايات التدريب مع تهيئة معينة).

---


## 7. جدول مقارنة

### RMSNorm مقابل LayerNorm

| الخاصية | LayerNorm | RMSNorm |
|---|---|---|
| **طرح المتوسط** | نعم | لا |
| **معاملات قابلة للتعلم** | $\gamma$ و$\beta$ | $\gamma$ فقط |
| **عدد العمليات الحسابية** | أكثر (حساب متوسط + تباين) | أقل (حساب RMS فقط) |
| **السرعة النسبية** | أبطأ قليلًا | أسرع (~10-15% عادة) |
| **الأداء التجريبي في LLMs** | جيد | مماثل أو أفضل قليلًا |
| **الاستخدام في النماذج الحديثة (2023+)** | نادر نسبيًا | شبه حصري (LLaMA، Mistral، إلخ) |

### مقارنة دوال التفعيل في FeedForward

| الخاصية | ReLU FFN | GELU FFN | SwiGLU |
|---|---|---|---|
| **عدد مصفوفات الأوزان** | 2 | 2 | 3 |
| **آلية البوابة (Gating)** | لا يوجد | لا يوجد | نعم (ضرب عنصري) |
| **الأداء التجريبي** | أساسي | جيد | الأفضل عادة |
| **التكلفة الحسابية الإضافية** | لا شيء | طفيفة | طفيفة (إسقاط إضافي) |
| **الاستخدام في النماذج الحديثة** | نادر جدًا الآن | شائع (BERT، GPT-2) | شبه حصري (LLaMA+) |

---


## 8. ملخص أفضل الممارسات

- ✅ استخدم RMSNorm وSwiGLU كخط أساس افتراضي لأي نموذج جديد تبنيه اليوم.
- ✅ اضبط دائمًا $d_{ff}$ بالنسبة الصحيحة (حوالي 2/3 من القيمة التقليدية) عند استخدام SwiGLU للحفاظ على ميزانية معاملات متسقة.
- ✅ لا تتردد في قياس الفرق الفعلي في بيئتك بدلًا من الاعتماد فقط على الأرقام المُبلَّغ عنها في الأوراق البحثية — قد تختلف المكاسب حسب العتاد وحجم النموذج.
- ✅ اجمع هذه اللبنات الصغيرة (RMSNorm + SwiGLU) مع الابتكارات الأكبر (RoPE + GQA) لبناء معمارية حديثة متكاملة كما في `ModernTransformerBlock`.

---


## 9. أسئلة مقابلات

**س1: لماذا يعمل RMSNorm بشكل جيد رغم تجاهله لخطوة طرح المتوسط التي تُعتبر جزءًا أساسيًا من فكرة "التطبيع" تقليديًا؟**

**الإجابة النموذجية**: الفرضية الأساسية وراء نجاح RMSNorm هي أن الفائدة الرئيسية لأي تقنية تطبيع في الشبكات العميقة تكمن في **التحكم في مقياس (scale)** التنشيطات بثبات عبر الطبقات (منع نموها الهائل أو تقلّصها نحو الصفر عبر عشرات أو مئات الطبقات المتتالية)، وليس بالضرورة في "إعادة التمركز" (re-centering) حول الصفر. تجريبيًا، أظهرت الدراسات (Zhang & Sennrich، 2019) أن معظم الفائدة العملية لـ LayerNorm تأتي من مكون القسمة على الانحراف المعياري (الذي يتحكم في المقياس)، بينما مساهمة طرح المتوسط في تحسين استقرار التدريب أو الأداء النهائي محدودة نسبيًا. بإزالة خطوة طرح المتوسط، يُصبح RMSNorm أبسط حسابيًا وأسرع (عملية حسابية واحدة أقل لكل تطبيع)، دون التضحية بشكل ملموس بالفائدة الأساسية للتطبيع، وهذا ما تؤكده التجارب الواسعة على نماذج اللغة الكبيرة الحديثة.

**س2: ما الفرق الجوهري بين طبقة FeedForward التقليدية وSwiGLU، ولماذا تحتاج SwiGLU لتعديل بُعد الطبقة الداخلية $d_{ff}$؟**

**الإجابة النموذجية**: طبقة FeedForward التقليدية تتكوّن من مصفوفتي وزن فقط: إسقاط للأعلى (`Linear1`) تليه دالة تفعيل غير خطية (مثل ReLU أو GELU)، ثم إسقاط للأسفل (`Linear2`) للعودة للبُعد الأصلي. SwiGLU، باعتبارها نوعًا من Gated Linear Units، تُدخل آلية "بوابة" ضربية: تُحسَب مسارين متوازيين من المدخل نفسه — أحدهما يمر بدالة SiLU (`SiLU(xW1)`) ويلعب دور "البوابة" التي تتحكم بكمية المعلومات المسموح لها بالمرور، بينما الآخر (`xW3`) يحمل "المحتوى" الفعلي؛ يُضرَب الاثنان عنصريًا، ثم يُمرَّر الناتج عبر إسقاط أخير (`W2`) للعودة للبُعد الأصلي. هذا التصميم يتطلب **ثلاث** مصفوفات أوزان ($W_1, W_2, W_3$) بدلًا من اثنتين فقط. لضمان أن إجمالي عدد المعاملات في طبقة SwiGLU لا يتجاوز بشكل كبير إجمالي معاملات طبقة FeedForward التقليدية المكافئة (بنفس $d_{ff}$)، تُطبِّق المعماريات الحديثة (مثل LLaMA) تصحيحًا يُقلِّص قيمة $d_{ff}$ إلى حوالي ثلثي قيمتها التقليدية (4× d_model)، بحيث يبقى إجمالي عدد المعاملات الكلي للطبقة مماثلًا تقريبًا، بينما تستفيد من آلية البوابة الإضافية التي أظهرت تحسينات تجريبية ثابتة في جودة النموذج.

---


## 10. تمرين عملي

1. نفّذ `RMSNorm` و`SwiGLUFeedForward` وتحقق من أن أشكال المدخل والمخرج متطابقة.
2. احسب يدويًا (أو بكود بسيط) RMSNorm لمتجه بمتوسط غير صفري عمدًا (مثلًا $[10, 12, 8, 14]$)، وقارن النتيجة مع LayerNorm لنفس المتجه — لاحظ الفرق هذه المرة بوضوح.
3. ابنِ `ModernTransformerBlock` كاملة، واحسب عدد معاملاتها الإجمالي، وقارنه بعدد معاملات `TransformerEncoderLayer` التقليدية (من الفصل 11) بنفس القيم تقريبًا لـ d_model وd_ff — لاحظ التأثير الصافي لكل من RMSNorm وSwiGLU وGQA مجتمعة.
4. شغّل `compare_norm_speed` على بيئتك وسجّل نسبة التسريع الفعلية لـ RMSNorm.

---


## 11. مراجع للتعمّق

- Zhang, B., & Sennrich, R. (2019). *Root Mean Square Layer Normalization*.
- Shazeer, N. (2020). *GLU Variants Improve Transformer*.
- Touvron, H., et al. (2023). *LLaMA: Open and Efficient Foundation Language Models*. (التطبيق العملي الكامل لكل هذه اللبنات مجتمعة)


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — المعماريات الحديثة / الفصل 17: المكوّنات الحديثة: RMSNorm وSwiGLU.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
